# Vehicle Detector — Approach 1: ResNet-18 Fine-tuning

**Task**: Binary classification — *is a vehicle visible in the frame?*  
**Dataset**: CARLA driving simulator (front-facing RGB camera)  
**Backbone**: ResNet-18 pretrained on ImageNet  
**Strategy**: Phase 1 — frozen backbone, train head only → Phase 2 — full differential fine-tune  
**Loss**: BCEWithLogitsLoss with class-frequency pos_weight  
**Note**: Vehicle class is dominant (~90 %+ positive rate) — pos_weight will be < 1.  
**Environment**: Google Colab Pro · A100 GPU (40 GB)  

---

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch  : {torch.__version__}  |  CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.set_float32_matmul_precision('high')

In [ ]:
# !pip install -q scikit-learn matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    roc_curve, precision_score, recall_score,
)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
TARGET    = 'has_vehicle'
TASK_NAME = 'Vehicle'

DRIVE_ROOT = Path('/content/drive/MyDrive/ML_Safety_2026')
TRAIN_DIR  = DRIVE_ROOT / 'train'      / 'train'
VAL_DIR    = DRIVE_ROOT / 'validation' / 'validation'
TEST_DIR   = DRIVE_ROOT / 'test'       / 'test'
CKPT_DIR   = DRIVE_ROOT / 'checkpoints' / TASK_NAME / 'ResNet18'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE        = 224
BATCH_SIZE      = 256
NUM_WORKERS     = 4
EPOCHS_FROZEN   = 5
EPOCHS_FINETUNE = 20
LR_HEAD         = 1e-3
LR_BACKBONE_FT  = 3e-5
LR_HEAD_FT      = 1e-4
WEIGHT_DECAY    = 1e-4
THRESHOLD       = 0.5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  Task: {TASK_NAME}  ({TARGET})')

In [ ]:
class CarlaDataset(Dataset):
    def __init__(self, split_dir, target_col, transform=None):
        split_dir = Path(split_dir)
        df = pd.read_csv(split_dir / 'labels.csv')
        df['frame'] = df['frame'].astype(str).str.zfill(6)
        df['img_path'] = df['frame'].apply(
            lambda f: str(split_dir / 'rgb-front' / f'{f}.jpg')
        )
        exists = df['img_path'].apply(os.path.exists)
        if not exists.all():
            print(f'  {(~exists).sum()} images missing — dropping.')
        df = df[exists].reset_index(drop=True)
        df[target_col] = df[target_col].map(
            {True: 1, False: 0, 'True': 1, 'False': 0}
        ).astype(int)
        self.df = df; self.target_col = target_col; self.transform = transform

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(float(row[self.target_col]), dtype=torch.float32)

    def pos_weight(self):
        v   = self.df[self.target_col].values
        neg, pos = (v == 0).sum(), (v == 1).sum()
        print(f'  Negative: {neg:,}  Positive: {pos:,}  ratio {neg/max(pos,1):.2f}:1')
        return torch.tensor([neg / max(pos, 1)], dtype=torch.float32)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print('Building datasets ...')
train_ds = CarlaDataset(TRAIN_DIR, TARGET, train_tf)
val_ds   = CarlaDataset(VAL_DIR,   TARGET, val_tf)
test_ds  = CarlaDataset(TEST_DIR,  TARGET, val_tf)
print('Train:'); pw = train_ds.pos_weight()
print(f'  {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw)

In [ ]:
def denorm(t):
    t = t.permute(1, 2, 0).numpy()
    return np.clip(t * np.array(STD) + np.array(MEAN), 0, 1)

vis_ds = CarlaDataset(TRAIN_DIR, TARGET, val_tf)
idxs   = random.sample(range(len(vis_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, idx in enumerate(idxs):
    img, lbl = vis_ds[idx]
    ax = axes[i // 4][i % 4]
    ax.imshow(denorm(img))
    color = 'limegreen' if lbl.item() == 1 else 'tomato'
    ax.set_title('Present' if lbl.item() == 1 else 'Absent',
                 color=color, fontweight='bold', fontsize=11)
    ax.axis('off')
plt.suptitle(f'Sample frames — {TASK_NAME}', fontsize=14)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'samples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
def build_resnet18(freeze_backbone=True):
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    in_feat = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(in_feat, 1))
    return model

model = build_resnet18(freeze_backbone=True).to(DEVICE)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_train:,} trainable / {n_total:,} total ({100*n_train/n_total:.1f}%)')

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pw.to(DEVICE))
scaler    = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    tot_loss, preds_all, labels_all = 0.0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(imgs).squeeze(1)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        tot_loss += loss.item() * len(labels)
        p = (torch.sigmoid(logits) > THRESHOLD).long().cpu().numpy()
        preds_all.extend(p); labels_all.extend(labels.long().cpu().numpy())
    scheduler.step()
    return tot_loss / len(loader.dataset), f1_score(labels_all, preds_all, zero_division=0)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss, probs_all, preds_all, labels_all = 0.0, [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(imgs).squeeze(1)
            loss   = criterion(logits, labels)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        tot_loss += loss.item() * len(labels)
        probs_all.extend(probs); preds_all.extend((probs > THRESHOLD).astype(int))
        labels_all.extend(labels.long().cpu().numpy())
    la = np.array(labels_all)
    return dict(
        loss=tot_loss/len(loader.dataset), acc=accuracy_score(la,preds_all),
        f1=f1_score(la,preds_all,zero_division=0),
        auc=roc_auc_score(la,probs_all) if len(np.unique(la))>1 else 0.5,
        prec=precision_score(la,preds_all,zero_division=0),
        rec=recall_score(la,preds_all,zero_division=0),
        probs=probs_all, labels=labels_all,
    )

In [ ]:
history  = {k: [] for k in ('tr_loss','tr_f1','val_loss','val_f1','val_auc')}
best_f1  = 0.0; best_auc = 0.0

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FROZEN, eta_min=1e-6)

print('Phase 1: Head-only  ─' + '─' * 45)
for ep in range(1, EPOCHS_FROZEN + 1):
    tl, tf = train_epoch(model, train_loader, optimizer, scheduler)
    vm     = evaluate(model, val_loader)
    for k, v in [('tr_loss',tl),('tr_f1',tf),('val_loss',vm['loss']),
                 ('val_f1',vm['f1']),('val_auc',vm['auc'])]: history[k].append(v)
    mark = ''
    if vm['f1'] >= best_f1:
        best_f1 = vm['f1']; best_auc = vm['auc']
        torch.save(model.state_dict(), CKPT_DIR / 'best_model.pth'); mark = '  ✓'
    print(f'Ep {ep:02d}/{EPOCHS_FROZEN}  tr {tl:.4f}/{tf:.4f}  '
          f'val {vm["loss"]:.4f}/{vm["f1"]:.4f}/{vm["auc"]:.4f}{mark}')
print(f'Phase 1 best → F1 {best_f1:.4f}  AUC {best_auc:.4f}')

In [ ]:
for p in model.parameters(): p.requires_grad = True
param_groups = [
    {'params': [p for n, p in model.named_parameters() if 'fc' not in n], 'lr': LR_BACKBONE_FT},
    {'params': model.fc.parameters(), 'lr': LR_HEAD_FT},
]
optimizer = optim.Adam(param_groups, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINETUNE, eta_min=1e-7)

print('Phase 2: Full fine-tuning  ─' + '─' * 37)
for ep in range(1, EPOCHS_FINETUNE + 1):
    tl, tf = train_epoch(model, train_loader, optimizer, scheduler)
    vm     = evaluate(model, val_loader)
    for k, v in [('tr_loss',tl),('tr_f1',tf),('val_loss',vm['loss']),
                 ('val_f1',vm['f1']),('val_auc',vm['auc'])]: history[k].append(v)
    mark = ''
    if vm['f1'] >= best_f1:
        best_f1 = vm['f1']; best_auc = vm['auc']
        torch.save(model.state_dict(), CKPT_DIR / 'best_model.pth'); mark = '  ✓'
    print(f'Ep {EPOCHS_FROZEN+ep:02d}/{EPOCHS_FROZEN+EPOCHS_FINETUNE}  '
          f'tr {tl:.4f}/{tf:.4f}  val {vm["loss"]:.4f}/{vm["f1"]:.4f}/{vm["auc"]:.4f}{mark}')
print(f'Best overall → F1 {best_f1:.4f}  AUC {best_auc:.4f}')

In [ ]:
T  = EPOCHS_FROZEN + EPOCHS_FINETUNE
ep = range(1, T + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vline_kw  = dict(x=EPOCHS_FROZEN+0.5, ls='--', color='gray', alpha=0.6, label='Unfreeze')
axes[0].plot(ep, history['tr_loss'], lw=2, label='Train')
axes[0].plot(ep, history['val_loss'], lw=2, label='Val')
axes[0].axvline(**vline_kw); axes[0].set(title='Loss', xlabel='Epoch'); axes[0].legend()
axes[1].plot(ep, history['tr_f1'], lw=2, label='Train')
axes[1].plot(ep, history['val_f1'], lw=2, label='Val')
axes[1].axvline(**vline_kw); axes[1].set(title='F1', xlabel='Epoch', ylim=[0,1]); axes[1].legend()
axes[2].plot(ep, history['val_auc'], color='darkorange', lw=2)
axes[2].axvline(**vline_kw); axes[2].set(title='Val AUC-ROC', xlabel='Epoch', ylim=[0,1])
plt.suptitle(f'Training Curves — {TASK_NAME} (ResNet-18)', fontsize=14)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best_model.pth', map_location=DEVICE))
tm = evaluate(model, test_loader)
print(f'=== Test Results — {TASK_NAME} (ResNet-18) ===')
for k in ('acc','f1','prec','rec','auc'): print(f'  {k:10s}: {tm[k]:.4f}')
print()
print(classification_report(tm['labels'], [int(p>THRESHOLD) for p in tm['probs']],
                            target_names=['Absent','Present']))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(tm['labels'], [int(p>THRESHOLD) for p in tm['probs']])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Absent','Present'], yticklabels=['Absent','Present'])
axes[0].set(title=f'Confusion Matrix — {TASK_NAME}', xlabel='Predicted', ylabel='True')
fpr, tpr, _ = roc_curve(tm['labels'], tm['probs'])
axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {tm["auc"]:.4f}')
axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
axes[1].set(title='ROC Curve', xlabel='FPR', ylabel='TPR', xlim=[0,1], ylim=[0,1])
axes[1].legend(fontsize=11)
plt.suptitle(f'Test Evaluation — {TASK_NAME} (ResNet-18)', fontsize=14)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'test_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results = {
    'approach': 'Approach 1 — ResNet-18 Standard Fine-tuning', 'target': TARGET, 'task': TASK_NAME,
    'hparams': dict(img_size=IMG_SIZE, batch_size=BATCH_SIZE, epochs_frozen=EPOCHS_FROZEN,
                    epochs_finetune=EPOCHS_FINETUNE, weight_decay=WEIGHT_DECAY),
    'test': dict(accuracy=round(tm['acc'],4), f1=round(tm['f1'],4),
                 precision=round(tm['prec'],4), recall=round(tm['rec'],4), auc=round(tm['auc'],4)),
    'val_best_f1': round(best_f1,4), 'val_best_auc': round(best_auc,4),
}
out = CKPT_DIR / 'results.json'
with open(out, 'w') as f: json.dump(results, f, indent=2)
print(f'Saved → {out}')
print(json.dumps(results, indent=2))